# Unit 3 — Fast Enough?

A contest program can find the right answer and still lose. If it makes the judge wait past the time limit, the judge rejects it. Today we will learn to estimate how much work an approach does before we spend time coding it.

## Count the Operations

We do not need to time a program with a stopwatch to compare two ideas. We can count an important repeated operation. A loop that visits each of `N` values once does about `N` operations.

In [ ]:
values = [8, 3, 6, 1, 9]
single_loop_checks = 0
for value in values:
    single_loop_checks = single_loop_checks + 1

print("N:", len(values))
print("Single-loop checks:", single_loop_checks)
assert single_loop_checks == 5

A loop inside another loop is different. For each of the `N` outer trips, the inner loop makes `N` trips. That gives about `N * N`, or `N²`, operations.

In [ ]:
values = [8, 3, 6, 1, 9]
nested_loop_checks = 0
for first in values:
    for second in values:
        nested_loop_checks = nested_loop_checks + 1

print("N:", len(values))
print("Nested-loop checks:", nested_loop_checks)
assert nested_loop_checks == 25

## Three Growth Families

**O(n)** means the work grows roughly with the input size. Double `N`, and the work roughly doubles. One full scan is the usual picture.

**O(n²)** means the work grows roughly like `N * N`. Double `N`, and the work grows to about four times as much. A full nested scan is the usual warning sign.

**O(log n)** means each step throws away a large part of what remains. Repeatedly cutting a numeric range in half takes surprisingly few steps, even when the starting range is huge.

Big-O describes the shape of growth, not an exact stopwatch time.

In [ ]:
for n in [10, 100, 1000]:
    linear_work = n
    square_work = n * n
    print("N =", n, "O(n) picture =", linear_work, "O(n squared) picture =", square_work)

## Read the Constraints Before Choosing

Problem constraints tell us the largest input the judge may use. Design for that largest case, not only for the tiny sample.

If `N` is up to 100, an O(n²) approach makes about `100 * 100 = 10,000` checks. That is usually fine.

If `N` is up to 100,000, an O(n²) approach could make about `100,000 * 100,000 = 10,000,000,000` checks. That is far too much for a typical contest time limit, so we need an approach close to O(n).

In [ ]:
small_n = 100
large_n = 100000
print("Small O(n squared) estimate:", small_n * small_n)
print("Large O(n squared) estimate:", large_n * large_n)
print("Large O(n) estimate:", large_n)
assert small_n * small_n == 10000
assert large_n * large_n == 10000000000

## Halve a Number Range

Imagine a secret whole number from 1 through 100. Guess the middle of the possible range. If the secret is higher, keep only the upper half; if it is lower, keep only the lower half. This is a numeric-range guessing walkthrough, not a search through a list.

Each guess cuts the remaining range about in half. That shrinking pattern is O(log n).

In [ ]:
secret = 73
low = 1
high = 100
guesses = 0

while low <= high:
    guess = (low + high) // 2
    guesses = guesses + 1
    print("Guess", guesses, "tries", guess, "inside", low, "through", high)
    if guess == secret:
        break
    elif guess < secret:
        low = guess + 1
    else:
        high = guess - 1

print("Found it in", guesses, "guesses")
assert guess == 73
assert guesses == 6

## One Problem, Two Approaches

Problem: given `N`, a target `K`, and `N` numbers, decide whether two different positions hold numbers that add to `K`. Return `YES` or `NO`. For this demonstration, the returned text also reports how many pair checks the approach made.

The direct idea tries each position with every later position. It is easy to understand, but it can make about N² checks.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    numbers = []
    for position in range(n):
        numbers.append(int(tokens[position + 2]))

    checks = 0
    found = False
    for first_position in range(n):
        for second_position in range(first_position + 1, n):
            checks = checks + 1
            if numbers[first_position] + numbers[second_position] == target:
                found = True

    answer = "NO"
    if found:
        answer = "YES"
    return answer + "\nPair checks: " + str(checks)

slow_result = solve("6 20\n2 4 6 8 10 12")
print(slow_result)
assert slow_result == "YES\nPair checks: 15"

The faster idea remembers numbers already visited in a dictionary. For each new number, it asks whether the needed partner is already there. One pass through `N` numbers makes `N` partner checks, so this approach is O(n).

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    seen = {}
    checks = 0
    found = False

    for position in range(n):
        number = int(tokens[position + 2])
        needed = target - number
        checks = checks + 1
        if needed in seen:
            found = True
        seen[number] = True

    answer = "NO"
    if found:
        answer = "YES"
    return answer + "\nPartner checks: " + str(checks)

fast_result = solve("6 20\n2 4 6 8 10 12")
print(fast_result)
assert fast_result == "YES\nPartner checks: 6"

Both approaches answer the same problem correctly. On six values, the direct version made 15 pair checks and the single-pass version made 6 partner checks. The gap becomes enormous near `N = 100,000`.

## A Contest Checklist

Before coding, ask: What is the largest `N`? How many times does my code visit each value? Is there a loop that repeats a full scan? Could one running total, running best, dictionary, or one in-place sort followed by one scan do the job?

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper. It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))